# Run 5 — execution-guided DPO on top of run 4

Four supervised runs all landed inside noise against run 1's 84.0%, so more data
is not the lever. Current work points at preference training as the largest one:
ExCoT-DPO reports BIRD 57.4 to 68.5 and Spider 78.8 to 86.6 after SFT.

Supervised fine-tuning only ever shows the model correct SQL, so it never learns
what its own mistakes look like. DPO does: each pair here puts the reference
query beside a specific wrong one the model actually produced, graded by
executing both against the same database.

**This run changes exactly one thing.** Same adapter, same data, same evaluation
as run 4 — only DPO is added. The previous runs varied several things at once
and the results were uninterpretable.

**Sidebar:** Accelerator `GPU T4 x2`, Internet **On**. Attach the dataset
`gb1105/qwen3-text2sql-run4-dpo` via **Add Input**.

In [ ]:
# --- 1. Hardware -----------------------------------------------------------
import os
# Kaggle gives two T4s and the libraries will split a 4B model across both,
# which only adds PCIe traffic. Pin to one before torch is imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import subprocess, torch
name = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
major, minor = torch.cuda.get_device_capability()
print(f"GPU         : {name}")
print(f"capability  : {major}.{minor}   visible: {torch.cuda.device_count()}")
if major < 7:
    raise SystemExit("STOP. Set Accelerator to 'GPU T4 x2' in the sidebar and Run All again.")
print("ok")

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets

In [ ]:
# --- 2. Verify the install, and that TRL has DPO ----------------------------
try:
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import DPOTrainer, DPOConfig
    import trl, peft, transformers
    print(f"ok | transformers {transformers.__version__} | trl {trl.__version__} | peft {peft.__version__}")
except Exception as e:
    print("INSTALL FAILED:", type(e).__name__, e)
    print("\nFallback, then Run > Restart session and skip the install cell:")
    print("  !pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo trl")

In [ ]:
# --- 3. Code and inputs ----------------------------------------------------
import glob, shutil, subprocess
from pathlib import Path

WORK = Path("/kaggle/working/ft")
if not (WORK / "evaluate.py").exists():
    shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
    subprocess.run(["git","clone","--depth","1","https://github.com/garvbahl37-gif/text2sql-qwen3-finetune.git","/kaggle/working/repo"], check=True)
    shutil.copytree("/kaggle/working/repo/training", WORK, dirs_exist_ok=True)
os.chdir(WORK)

hits = glob.glob("/kaggle/input/**/dpo_pairs.jsonl", recursive=True)
if not hits:
    raise SystemExit("Attach the dataset gb1105/qwen3-text2sql-run4-dpo with Add Input.")
INPUT = Path(hits[0]).parent
print("inputs:", sorted(p.name for p in INPUT.iterdir()))

def step(cmd: str):
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise SystemExit(f"Step failed (exit {p.returncode}): {cmd}")
    print("\nok\n", flush=True)

## 4. Preference training

The pairs were built locally: run 4 sampled at temperature 0.8 on 2,500 training
questions, each answer executed against the reference, and the genuine failures
kept. 549 pairs — 389 that ran and returned the wrong rows, 160 that did not run.

`beta` controls how far the policy may drift from the reference model. 0.1 is
the usual starting point; lower drifts further.

In [ ]:
# --- 4. DPO ----------------------------------------------------------------
import json
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import DPOTrainer, DPOConfig

SYSTEM = ("You are a precise text-to-SQL engine. Given a SQLite schema and a question, "
          "reply with a single valid SQLite query that answers it. Output only the SQL. "
          "No explanation, no markdown, no commentary.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=1024, dtype=None, load_in_4bit=True)

# continue from run 4 rather than starting over: DPO refines an SFT model
model.load_adapter(str(INPUT), adapter_name="default")
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

pairs = [json.loads(l) for l in open(INPUT / "dpo_pairs.jsonl")]
def fmt(p):
    prompt = tokenizer.apply_chat_template(
        [{"role":"system","content":SYSTEM},{"role":"user","content":p["prompt"]}],
        tokenize=False, add_generation_prompt=True)
    return {"prompt": prompt, "chosen": p["chosen"], "rejected": p["rejected"]}
ds = Dataset.from_list([fmt(p) for p in pairs])
print(f"{len(ds)} preference pairs")

trainer = DPOTrainer(
    model=model, ref_model=None, processing_class=tokenizer, train_dataset=ds,
    args=DPOConfig(
        output_dir="outputs/run5-dpo",
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=1, learning_rate=5e-6, lr_scheduler_type="cosine",
        warmup_ratio=0.1, beta=0.1, max_length=1024, max_prompt_length=896,
        optim="adamw_8bit", logging_steps=10,
        bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
        report_to="none", seed=42))
trainer.train()
model.save_pretrained("outputs/run5-dpo"); tokenizer.save_pretrained("outputs/run5-dpo")
print("saved outputs/run5-dpo")

## 5. Evaluate

The same two sets run 4 was scored on. The general set is what run 1 scored
84.0% on, so it says whether DPO actually moved anything; the hand-written
multi-level set says whether the capability survived.

In [ ]:
step(f"cp {INPUT}/test.jsonl data/test.jsonl && cp {INPUT}/multilevel_holdout.jsonl data/multilevel_holdout.jsonl")
step("python evaluate.py --adapter outputs/run5-dpo --test data/test.jsonl --limit 300 "
     "--out outputs/run5-general/eval_report.json")
step("python evaluate.py --adapter outputs/run5-dpo --test data/multilevel_holdout.jsonl "
     "--limit 18 --out outputs/run5-holdout/eval_report.json")

In [ ]:
# --- 6. Results ------------------------------------------------------------
import json, shutil
from pathlib import Path
out = Path("/kaggle/working")
RUN1, RUN4 = 0.840, 0.827   # both measured on this same 300-example set

for label, path, bar in (("GENERAL", "outputs/run5-general/eval_report.json", RUN4),
                         ("HAND-WRITTEN MULTI-LEVEL", "outputs/run5-holdout/eval_report.json", 0.278)):
    r = json.loads(Path(path).read_text())
    b, t = r["metrics"]["base"], r["metrics"]["tuned"]
    print(f"\n=== {label} — n={r['n_examples']} ===")
    for k in b:
        print(f"  {k:<22}{b[k]:>9.1%}{t[k]:>10.1%}")
    got = t["execution_accuracy"]
    print(f"  vs run 4 ({bar:.1%}): {(got-bar)*100:+.1f} points")
    if label == "GENERAL":
        print(f"  vs run 1 ({RUN1:.1%}): {(got-RUN1)*100:+.1f} points")
    shutil.copy(path, out / (Path(path).parent.name + ".json"))

shutil.make_archive(str(out / "run5-adapter"), "zip", "outputs/run5-dpo")
print("\nSaved to /kaggle/working — download from the Output panel.")